In [1]:
import os
import zipfile
import pandas as pd
from tqdm import tqdm

import torch
import torch_musa
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights

from dataset.spectrogram_dataset import SpectrogramDataset

Error in cpuinfo: prctl(PR_SVE_GET_VL) failed


In [2]:
class AudioNet(nn.Module):
    def __init__(self):
        super().__init__()
        model = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )  # pretrained weights on ImageNet
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Linear(model.fc.in_features, 2)
        self.model = model

    def forward(self, x):
        return self.model(x)

In [3]:
def train_one_epoch(model, train_loader, val_loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="Train"):
        x = batch["spectrogram"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f"Train Loss: {train_loss:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Val Split"):
            x = batch["spectrogram"].to(device)
            y = batch["label"].to(device)
            output = model(x)
            loss = criterion(output, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print(f"Val Split Loss: {val_loss:.4f}")

In [4]:
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Test"):
            x = batch["spectrogram"].to(device)
            output = model(x)
            pred = torch.argmax(output, dim=1)
            preds.extend(pred.cpu().numpy())
    return preds

In [5]:
def save_submission_csv(preds, save_name):
    df = pd.DataFrame(preds)
    df.to_csv(save_name, index=False, header=False)

In [6]:
device = torch.device("musa" if torch.musa.is_available() else "cpu")
model = AudioNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/dream/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████████████████████████████████| 44.7M/44.7M [00:08<00:00, 5.41MB/s]


In [7]:
full_train_set = SpectrogramDataset("dataset/training_set")

val_size = int(0.2 * len(full_train_set))
train_size = len(full_train_set) - val_size
train_set, val_split_set = random_split(full_train_set, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32)
val_split_loader = DataLoader(val_split_set, batch_size=32)

train_one_epoch(model, train_loader, val_split_loader, criterion, optimizer, device)

Train: 100%|████████████████████████████████| 1425/1425 [04:31<00:00,  5.24it/s]


Train Loss: 0.1821


Val Split: 100%|██████████████████████████████| 357/357 [00:28<00:00, 12.72it/s]

Val Split Loss: 0.1534


## 测试阶段

In [8]:
val_set = SpectrogramDataset("dataset/validation_set")
test_set = SpectrogramDataset("dataset/testing_set")

val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

val_preds = predict(model, val_loader, device)
test_preds = predict(model, test_loader, device)

Test: 100%|███████████████████████████████████| 223/223 [00:15<00:00, 14.49it/s]


In [9]:
save_submission_csv(val_preds, "submissionA.csv")
save_submission_csv(test_preds, "submissionB.csv")
with zipfile.ZipFile("submission.zip", "w") as zipf:
    zipf.write("submissionA.csv")
    zipf.write("submissionB.csv")
os.remove("submissionA.csv")
os.remove("submissionB.csv")